# Imported street weather notebook

This is the older street-weather notebook from `GITHUB_NEWER`. It is kept for reference.


In [13]:
import pandas as pd

cscl = pd.read_csv('CSCL.csv')

from shapely import wkt
import numpy as np

# Define station coordinates
stations = np.array([
    [40.703, -74.006],  # 1 City Hall
    [40.597, -73.916],  # 2 Brooklyn Salt Marsh
    [40.649, -73.790],  # 3 JFK
    [40.774, -73.879],  # 4 LGA
    [40.849, -73.878],  # 5 Bronx Zoo
    [40.588, -74.140],  # 6 Staten Island
])

def nearest_station(the_geom_wkt):
    # Parse geometry
    geom = wkt.loads(the_geom_wkt)

    # Extract one representative coordinate
    # (e.g., first coordinate of first LineString)
    if geom.geom_type == "MultiLineString":
        coord = geom.geoms[0].coords[0]
    elif geom.geom_type == "LineString":
        coord = geom.coords[0]
    else:
        return None  # skip invalid geometry

    lat, lon = coord[1], coord[0]  # assuming WKT order is (x, y) = (lon, lat)
    
    # Compute Euclidean distance in degrees
    # (sufficient for NYC-scale comparisons)
    dists = np.sqrt((stations[:,0] - lat)**2 + (stations[:,1] - lon)**2)
    
    # Return station index (1-based)
    return int(np.argmin(dists) + 1)

# Apply to DataFrame
cscl['nearest_station'] = cscl['the_geom'].apply(nearest_station)


In [2]:
# NORMALIZATION

import re

ABBREV = {
    'st': 'street', 'st.': 'street', 'street.': 'street',
    'ave': 'avenue', 'av': 'avenue', 'av.': 'avenue',
    'rd': 'road', 'rd.': 'road',
    'blvd': 'boulevard', 'blvd.': 'boulevard',
    'pl': 'place', 'plz': 'plaza', 'pl.': 'place',
    'ct': 'court', 'ctr': 'center',
    'ln': 'lane', 'dr': 'drive', 'ter': 'terrace',
    'hwy': 'highway', 'pkwy': 'parkway',
    'sq': 'square', 'e': 'east', 'e.': 'east', 'w': 'west', 'w.': 'west',
    's': 'south', 's.': 'south', 'n': 'north', 'n.': 'north',
    'wash': 'washington', 'wash.': 'washington', 'ft': 'fort',
    'pl': 'place', 'pl.': 'place', 'aly': 'alley', 'aly.': 'alley',
    'cres': 'crescent', 'cres.': 'crescent', 'cr': 'crescent', 'cr.': 'crescent',
    'cir': 'circle', 'cir.': 'circle', 'grn': 'green', 'grn.': 'green',
    'hl': 'hill', 'hl.': 'hill', 'mt': 'mount', 'mt.': 'mount'
}

ORDINAL = {
    'first': '1', '1st': '1',
    'second': '2', '2nd': '2',
    'third': '3', '3rd': '3',
    'fourth': '4', '4th': '4',
    'fifth': '5', '5th': '5',
    'sixth': '6', '6th': '6',
    'seventh': '7', '7th': '7',
    'eighth': '8', '8th': '8',
    'ninth': '9', '9th': '9',
    'tenth': '10', '10th': '10'
}

# --- Alias placeholder (populate later) ---
ALIASES = {
        'ave of the americas': '6 ave',
    'avenue of the americas': '6 ave',
    'west 110 street': '110 st',
    'andrews avenue north': 'andrews avenue', 'andrews avenue south': 'andrews avenue'
}

# --- Normalization function ---
def normalize(name: str) -> str:
    """Canonicalize street name with lowercase, punctuation removal,
       abbreviation/ordinal expansion, and alias replacement."""
    if not isinstance(name, str):
        return ''
    s = name.lower()
    if s in ALIASES:
        s = ALIASES[s]
    s = re.sub(r'[^a-z0-9\s]', ' ', s)
    s = re.sub(r'\s+', ' ', s).strip()
    words = []
    for w in s.split():
        if w in ORDINAL:
            w = ORDINAL[w]
        elif w in ABBREV:
            w = ABBREV[w]
        words.append(w)
    s_norm = ' '.join(words)

    return s_norm

In [3]:
cscl["normalized_street_name"] = (
    cscl["Borough Code"].astype(str)
    + "-"
    + cscl["Full Street Name"].apply(normalize)
)

In [4]:
import numpy as np
import pandas as pd

# --- 1. Compute median values grouped by type (either POST_TYPE or PRE_TYPE) ---
# Flatten POST_TYPE and PRE_TYPE into a single "type" for median calculation
cscl_types = cscl.copy()
cscl_types["TYPE"] = cscl_types.apply(
    lambda r: r["POST_TYPE"] if pd.notna(r["POST_TYPE"]) else r["PRE_TYPE"], axis=1
)

medians = (
    cscl_types.groupby("TYPE")[["Segment Length", "Street Width"]]
    .median(numeric_only=True)
)

# --- 2. Function to impute by type ---
def impute_by_type(row):
    # Determine the type to use
    types_present = []
    if pd.notna(row["POST_TYPE"]):
        types_present.append(row["POST_TYPE"])
    if pd.notna(row["PRE_TYPE"]):
        types_present.append(row["PRE_TYPE"])
    
    if len(types_present) == 1:
        t = types_present[0]
    elif len(types_present) == 2:
        if types_present[0] == types_present[1]:
            t = types_present[0]
        else:
            t = None  # conflict, cannot impute
    else:
        t = None  # neither populated
    
    if t and t in medians.index:
        if pd.isna(row["Segment Length"]):
            row["Segment Length"] = medians.loc[t, "Segment Length"]
        if pd.isna(row["Street Width"]):
            row["Street Width"] = medians.loc[t, "Street Width"]
    return row

# Apply imputation
cscl = cscl.apply(impute_by_type, axis=1)

# --- 3. Handle rows where type info missing but Segment Length or Street Width is missing ---
mask_missing_type = (cscl["POST_TYPE"].isna() & cscl["PRE_TYPE"].isna()) & (
    cscl["Segment Length"].isna() | cscl["Street Width"].isna()
)
problem_rows = cscl[mask_missing_type].copy()

In [5]:
cscl = cscl[["PHYSICALID", "normalized_street_name", "Segment Length", "Street Width", "nearest_station"]]

In [6]:
# dictionary normalized street name: all physical id's with that street name

name_to_ids = (
    cscl.groupby("normalized_street_name")["PHYSICALID"]
    .apply(list)
    .to_dict()
)

new df called street_weather constructed as follows: it has a "normalized_street_name" column, and columns "DATA 20XX" where DATA is in {temperature, apparent_temperature, snowfall, snow_depth, precipitation, relative_humidity, cloud_cover, wind_gusts_10m, wind_speed_10m}. normalized_street_name goes over all those keys from the dictionary name_to_ids. all other values are computed as the average value across (*) entries in the column whose name begins with that given string (e.g. temperature will be the start of actual string name "temperate_2m (degrees F)" in the corresponding csv) in weather_hourly_i where i is the value in nearest_station in cscl df for that normalized_street_name (might not be unique row, just choose any row with that normalized_street_name), where (*) means cover november 21 20XX-1 to march 20 20XX (e.g. november 21 2015 to march 20 2016 for 20XX = 2016), and 20XX goes from 2016 to 2025

In [7]:
import pandas as pd
import numpy as np
from pathlib import Path

# --- Config ---
weather_dir = Path(".")  # set to your actual path
YEARS = range(2016, 2026)
VARIABLES = [
    "temperature",
    "apparent_temperature",
    "snowfall",
    "snow_depth",
    "precipitation",
    "relative_humidity",
    "cloud_cover",
    "wind_gusts_10m",
    "wind_speed_10m",
]

# --- Load weather_hourly_i CSVs ---
weather_hourly = {}
for i in range(1, 7):
    f = weather_dir / f"weather_hourly_{i}.csv"
    df = pd.read_csv(f, skiprows=3)
    # Try to find datetime column heuristically
    datetime_col = next(c for c in df.columns if "time" in c.lower())
    df[datetime_col] = pd.to_datetime(df[datetime_col], errors="coerce")
    df.rename(columns={datetime_col: "datetime"}, inplace=True)
    weather_hourly[i] = df

# --- Helper to match a variable prefix ---
def get_column(df, prefix):
    prefix = prefix.lower()
    for c in df.columns:
        if c.lower().startswith(prefix):
            return c
    raise KeyError(f"No column starts with '{prefix}' in {df.columns[:8]}...")

# --- Map streets to nearest station ---
station_map = cscl.groupby("normalized_street_name")["nearest_station"].first().to_dict()

records = []

for street, ids in name_to_ids.items():
    if street not in station_map:
        continue

    station_id = station_map[street]
    if station_id not in weather_hourly:
        continue

    wdf = weather_hourly[station_id]

    for year in YEARS:
        start = pd.Timestamp(year - 1, 11, 21)
        end = pd.Timestamp(year, 3, 20, 23, 59)
        sub = wdf.loc[(wdf["datetime"] >= start) & (wdf["datetime"] <= end)]

        if sub.empty:
            continue

        entry = {"normalized_street_name": street}
        for var in VARIABLES:
            col = get_column(wdf, var)
            entry[f"{var} {year}"] = sub[col].mean(skipna=True)
        records.append(entry)

# --- Construct final DataFrame ---
street_weather = pd.DataFrame(records)
street_weather = street_weather.groupby("normalized_street_name").first().reset_index()


In [ ]:
# Melt wide columns (e.g. temperature 2016, snowfall 2016, …) into long form

street_weather_melted = street_weather.melt(
    id_vars=['normalized_street_name'],
    var_name='variable_season',
    value_name='value'
)

# Split variable and season
street_weather_melted[['variable', 'season']] = street_weather_melted['variable_season'].str.extract(r'(.+)\s(\d{4})')

# Create combined identifier
street_weather_melted['normalized_street_name_season'] = (
    street_weather_melted['normalized_street_name'] + '_' + street_weather_melted['season']
)

# Pivot back so that each variable becomes a column
street_weather_long = street_weather_melted.pivot_table(
    index='normalized_street_name_season',
    columns='variable',
    values='value',
    aggfunc='first'
).reset_index()

# Optional: flatten columns
street_weather_long.columns.name = None

# Result:
# columns → ['normalized_street_name_season', 'temperature', 'apparent_temperature', 'snowfall', ...]


In [12]:
street_weather_long.to_csv("street_weather.csv", index=False)